# 03. Проверка корректности эксперимента

## Содержание
1. [Анализ первого эксперимента](#анализ-первого-эксперимента)
2. [SRM-тест](#srm-тест)
3. [Аномалии по дням](#аномалии-по-дням)
4. [Пересечение групп](#пересечение-групп)
5. [Вердикт](#вердикт)
6. [Анализ второго эксперимента](#анализ-второго-эксперимента)

## Загрузка данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')
from src.data_loader import load_monitoring_data, load_results_data
from src.stats import srm_test

df_monitoring = load_monitoring_data()
df_results = load_results_data()

print(f"monitoring: {len(df_monitoring)} строк")
print(f"   Период: {df_monitoring['date'].min()} - {df_monitoring['date'].max()}")
print(f"results: {len(df_results)} строк")
print(f"   Период: {df_results['date'].min()} - {df_results['date'].max()}")

## Анализ первого эксперимента (мониторинг)

In [ ]:
# Общая статистика
total_control = len(df_monitoring[df_monitoring['group'] == 'control'])
total_treatment = len(df_monitoring[df_monitoring['group'] == 'treatment'])
total = len(df_monitoring)

print("Баланс групп (общий)\n")
print(f"Control:   {total_control} ({total_control/total*100:.1f}%)")
print(f"Treatment: {total_treatment} ({total_treatment/total*100:.1f}%)")
print(f"Ожидание:  50% / 50%")

In [ ]:
# SRM-тест
srm = srm_test(total_control, total_treatment)

print(f"\n📊 SRM-тест (Sample Ratio Mismatch)\n")
print(f"Chi-square: {srm['chi2']:.4f}")
print(f"p-value:    {srm['p_value']:.6f}")

if srm['is_valid']:
    print("SRM верен — распределение соответствует 50/50")
else:
    print("SRM нарушен — распределение НЕ соответствует 50/50")

In [ ]:
# Детальный анализ по дням
daily_balance = df_monitoring.groupby(['date', 'group']).size().unstack(fill_value=0)
daily_balance['total'] = daily_balance['control'] + daily_balance['treatment']
daily_balance['control_pct'] = daily_balance['control'] / daily_balance['total'] * 100
daily_balance['treatment_pct'] = daily_balance['treatment'] / daily_balance['total'] * 100
daily_balance['deviation'] = abs(daily_balance['control_pct'] - 50)

print("Баланс групп по дням\n")
print(daily_balance.round(1))

# Поиск аномалий
anomaly_days = daily_balance[daily_balance['deviation'] > 10]

if len(anomaly_days) > 0:
    print("\nАномальные дни (отклонение > 10% от 50/50):")
    print(anomaly_days[['control', 'treatment', 'control_pct', 'treatment_pct']])
else:
    print("\nВсе дни в пределах нормы")

In [ ]:
# Конверсия по дням
daily_conv = df_monitoring.groupby(['date', 'group'])['converted'].mean().unstack()

print("Конверсия по дням\n")
print("Дата       Control  Treatment  Diff")
print("-" * 45)

for date in daily_conv.index:
    c_conv = daily_conv.loc[date, 'control']
    t_conv = daily_conv.loc[date, 'treatment']
    diff = t_conv - c_conv
    print(f"{date.strftime('%d.%m')}  {c_conv:.4f}    {t_conv:.4f}    {diff:+.4f}")

In [ ]:
# Проверка уникальности
duplicates = df_monitoring[df_monitoring.duplicated(subset=['user_id'], keep=False)]
users_in_both = df_monitoring.groupby('user_id')['group'].nunique()
cross_users = users_in_both[users_in_both > 1]

print("🔍 Проверка уникальности пользователей\n")
print(f"Дубликатов user_id: {len(duplicates)}")
print(f"Пользователей в обеих группах: {len(cross_users)}")

if len(cross_users) > 0:
    print("Есть пользователи в обеих группах!")
else:
    print("Каждый пользователь только в одной группе")

In [ ]:
# Визуализация
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# График 1: Баланс групп
daily_balance[['control', 'treatment']].plot(kind='bar', ax=axes[0], color=['#2E86AB', '#E84855'], alpha=0.7)
axes[0].axhline(total/2, color='red', linestyle='--', label=f'Ожидание 50/50 ({total/2:.0f})')
axes[0].set_title('Баланс групп по дням (мониторинг)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Дата')
axes[0].set_ylabel('Количество пользователей')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# График 2: Конверсия
daily_conv.plot(marker='o', ax=axes[1], color=['#2E86AB', '#E84855'], linewidth=2)
axes[1].set_title('Конверсия по дням (мониторинг)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Дата')
axes[1].set_ylabel('Конверсия')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Вердикт по первому эксперименту

### Проблемы:

| Проблема | Серьезность | Описание |
|----------|-------------|----------|
| **SRM нарушен** | Критическая | p ≈ 0.000, распределение 64/36 |
| **Дисбаланс 04.03** | Критическая | 95.7% пользователей в control |
| **Нулевая конверсия** | Критическая | 03-04.03 CR = 0% в treatment |
| **Пересечение групп** | Серьезная | 164 пользователя в обеих группах |

### Вердикт:

> **Эксперимент НЕВАЛИДЕН. Требуется перезапуск.**

**Причины:**
1. Система сплитования работала некорректно
2. Технический сбой в трекинге конверсий
3. Нарушение независимости наблюдений

**Рекомендации:**
1. Остановить эксперимент
2. Исправить систему сплитования
3. Исправить трекинг конверсий
4. Перезапустить эксперимент с чистой выборкой

## Анализ второго эксперимента (валидный)

In [ ]:
# Общая статистика
total_control = len(df_results[df_results['group'] == 'control'])
total_treatment = len(df_results[df_results['group'] == 'treatment'])
total = len(df_results)

print("Баланс групп (второй эксперимент)\n")
print(f"Control:   {total_control} ({total_control/total*100:.1f}%)")
print(f"Treatment: {total_treatment} ({total_treatment/total*100:.1f}%)")
print(f"Ожидание:  50% / 50%")

In [ ]:
# SRM-тест
srm = srm_test(total_control, total_treatment)

print(f"\nSRM-тест\n")
print(f"Chi-square: {srm['chi2']:.4f}")
print(f"p-value:    {srm['p_value']:.6f}")

if srm['is_valid']:
    print("SRM верен — распределение соответствует 50/50")
else:
    print("SRM нарушен")

In [ ]:
# Детальный баланс
daily_balance = df_results.groupby(['date', 'group']).size().unstack(fill_value=0)
daily_balance['total'] = daily_balance['control'] + daily_balance['treatment']
daily_balance['control_pct'] = daily_balance['control'] / daily_balance['total'] * 100
daily_balance['deviation'] = abs(daily_balance['control_pct'] - 50)

anomaly_days = daily_balance[daily_balance['deviation'] > 10]

print("Баланс групп по дням\n")
print(daily_balance.round(1))

if len(anomaly_days) > 0:
    print("\nАномальные дни:")
    print(anomaly_days[['control', 'treatment', 'control_pct']])
else:
    print("\nВсе дни в пределах нормы")

In [ ]:
# Проверка уникальности
duplicates = df_results[df_results.duplicated(subset=['user_id'], keep=False)]
users_in_both = df_results.groupby('user_id')['group'].nunique()
cross_users = users_in_both[users_in_both > 1]

print("🔍 Проверка уникальности пользователей\n")
print(f"Дубликатов user_id: {len(duplicates)}")
print(f"Пользователей в обеих группах: {len(cross_users)}")

if len(cross_users) > 0:
    print("Есть пользователи в обеих группах!")
else:
    print("Каждый пользователь только в одной группе")

## Вывод по второму эксперименту

### Проверки:

| Проверка | Результат | Статус |
|----------|-----------|--------|
| **SRM-тест** | p = 1.000 | Пройден |
| **Аномальные дни** | Нет отклонений > 10% | Норма |
| **Пересечение групп** | 0 пользователей | Нет |
| **Дубликаты user_id** | 0 | Нет |

### Вердикт:

> **Эксперимент признан ВАЛИДНЫМ.**

**Обоснование:**
1. SRM пройден (p = 1.000)
2. Нет аномальных дней
3. Нет пересечений пользователей
4. Можно переходить к статистическому анализу